# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My rule and its reason codes

### Signal Audit Before Rule Encoding
Before writing a baseline rule, we test two underlying candidate signals against observed content decay (`trend_direction == 'down'`) in the starter dataset to verify whether our engineering assumptions hold:

1. **Signal 1: Staleness (`freshness_tier` / `days_since_last_update`)**
   - *Hypothesis behind FlyRank refresh flags:* Older content that has not been updated recently decays faster.
   - *Verdict:* **MIXED**.
   - *Rationale:* While content in the 91–180 day tier has a peak decline rate of 61.1%, content in the 181+ day tier drops back to 47.1% (survivor bias: very old pages that remain in the catalog are stable evergreen assets). Furthermore, freshly published/updated pages (0–30 days) already experience a 51.1% decline rate. Staleness alone is non-monotonic and insufficient as a standalone rule.

2. **Signal 2: Position Tier / SERP Vulnerability (`position_tier`)**
   - *Hypothesis behind FlyRank Page-One Decay / Striking Distance flags:* Pages ranked in striking distance (positions 11–20) and Page 1 (positions 4–10) face intense SERP competition and decay more rapidly than top-3 anchors or deep tail pages.
   - *Verdict:* **CONFIRMED**.
   - *Rationale:* Content in the `striking` tier (positions 11–20) has the highest decay rate at **60.95%**, closely followed by `page_1` at **56.97%**. In contrast, entrenched `top_3` content has only a **24.08%** decline rate, and deep tail content (positions 50+) has only **34.42%**. Ranking vulnerability is a genuine search signal.

---

### The Baseline Rule in Plain Words
> *"A content item is prioritized for refresh review if it commands significant search exposure (high impressions), occupies a competitively vulnerable ranking tier (Page 1 or striking distance, positions 1–20), and has reached maturity without recent editorial updates."*

### Reason Codes & Action Labels
- `page_one_decay_risk` -> Action: `refresh` (high impressions + Page 1 ranking).
- `striking_distance_opportunity` -> Action: `refresh_and_expand` (high impressions + positions 11–20).
- `stale_visible_page` -> Action: `refresh` (impressions >= 250 + age >= 180d).
- `high_volume_monitoring` -> Action: `monitor` (large volume but stable rank).


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robust path resolution for local, repo root, and Google Colab
data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
df_eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_eligible["is_declining"] = (df_eligible["trend_direction"] == "down").astype(int)

# --- Signal 1 Bucket Table: Staleness (freshness_tier) ---
s1_table = df_eligible.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining", "mean"),
    median_days=("days_since_last_update", "median")
).reset_index()

print("=== Signal 1 Check: Staleness vs. Content Decay ===")
print(s1_table.to_string(index=False))
print("Verdict: MIXED (Non-monotonic: 91-180d peaks at 61.1% decline, but 181+d drops to 47.1% due to evergreen survivor bias).")

# --- Signal 2 Bucket Table: Position Tier (position_tier) ---
s2_table = df_eligible.groupby("position_tier").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining", "mean"),
    median_pos=("avg_position", "median")
).reset_index()

print("\n=== Signal 2 Check: Position Tier vs. Content Decay ===")
print(s2_table.to_string(index=False))
print("Verdict: CONFIRMED (Striking distance [11-20] has peak decay at 61.0%, Page 1 [4-10] at 57.0%, while Top 3 is stable at 24.1%).")



=== Signal 1 Check: Staleness vs. Content Decay ===
freshness_tier     n  decline_rate  median_days
          0-30 20480      0.511377         20.0
          181+   174      0.471264        211.0
         31-90   175      0.588571         41.0
        91-180  9171      0.611057        104.0
Verdict: MIXED (Non-monotonic: 91-180d peaks at 61.1% decline, but 181+d drops to 47.1% due to evergreen survivor bias).

=== Signal 2 Check: Position Tier vs. Content Decay ===
position_tier     n  decline_rate  median_pos
         deep  1319      0.344200        61.0
       page_1 11814      0.569663         6.6
     page_3_5  7242      0.561585        28.9
     striking  7304      0.609529        13.9
        top_3  2321      0.240844         0.0
Verdict: CONFIRMED (Striking distance [11-20] has peak decay at 61.0%, Page 1 [4-10] at 57.0%, while Top 3 is stable at 24.1%).


## 2. Build the ranked queue (writes the CSV)

### Scoring Formula
We construct a transparent, hand-crafted baseline score $S \in [0, 100]$ using three pre-decision observable signals:
$$\text{baseline\_score} = \left( 0.40 \times \text{visibility} + 0.35 \times \text{position\_opportunity} + 0.25 \times \text{staleness} \right) \times 100$$
where:
- $\text{visibility} = \text{percentile\_rank}(\log(1 + \text{impressions\_90d}))$
- $\text{position\_opportunity} = \left(1 - \frac{\text{clip}(\text{avg\_position}, 1, 50) - 1}{49}\right) \times \mathbb{I}(\text{avg\_position} > 0)$
- $\text{staleness} = \text{percentile\_rank}(\text{days\_since\_last\_update})$

Each item receives ONE dominant reason code and an operational action label (`refresh`, `refresh_and_expand`, `monitor`). The ranked queue is written directly to `work/outputs/baseline_action_score.csv`.


In [2]:
from pathlib import Path
import json

# Percentile rank helper
def percentile_rank(series):
    return series.rank(pct=True)

# 1. Compute transparent score components
visibility = percentile_rank(np.log1p(df_eligible["impressions_90d"]))
pos_norm = (1 - (df_eligible["avg_position"].clip(1, 50) - 1) / 49.0) * (df_eligible["avg_position"] > 0).astype(int)
staleness = percentile_rank(df_eligible["days_since_last_update"])

df_eligible["baseline_score"] = (0.40 * visibility + 0.35 * pos_norm + 0.25 * staleness) * 100.0
df_eligible["baseline_score"] = df_eligible["baseline_score"].round(2)

# 2. Assign ONE primary reason code and action label
def assign_action_and_reason(row):
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["impressions_90d"] >= 500:
        return "page_one_decay_risk", "refresh"
    elif 10 < row["avg_position"] <= 20 and row["impressions_90d"] >= 500:
        return "striking_distance_opportunity", "refresh_and_expand"
    elif row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 250:
        return "stale_visible_page", "refresh"
    elif row["impressions_90d"] >= 1000:
        return "high_volume_monitoring", "monitor"
    else:
        return "general_opportunity", "monitor"

rule_assignments = [assign_action_and_reason(r) for _, r in df_eligible.iterrows()]
df_eligible["reason_code"] = [r[0] for r in rule_assignments]
df_eligible["action_label"] = [r[1] for r in rule_assignments]

# 3. Rank queue
df_eligible["rank"] = df_eligible["baseline_score"].rank(method="first", ascending=False).astype(int)
df_ranked = df_eligible.sort_values("rank").reset_index(drop=True)

# 4. Write output CSV to work/outputs/baseline_action_score.csv
# Robust output directory resolution for repo root, work/notebooks, and Colab
if Path("work/outputs").exists():
    out_dir = Path("work/outputs")
elif Path("../outputs").exists():
    out_dir = Path("../outputs")
else:
    out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "baseline_action_score.csv"

export_cols = [
    "rank", "content_id", "client_id", "baseline_score", "reason_code", 
    "action_label", "impressions_90d", "avg_position", "days_since_last_update", 
    "ctr", "trend_direction"
]
df_ranked[export_cols].to_csv(out_csv, index=False)
print(f"Successfully generated and wrote ranked queue to: {out_csv}")
print(f"Total rows exported: {len(df_ranked):,} | File size: {out_csv.stat().st_size:,} bytes")

# 5. Evaluate baseline precision metrics
y_true = (df_ranked["trend_direction"] == "down").astype(int).values
p10 = y_true[:10].mean()
p20 = y_true[:20].mean()
p50 = y_true[:50].mean()
base_rate = y_true.mean()

print(f"\n=== Baseline Rule Performance ===")
print(f"Base Decline Rate: {base_rate:.1%}")
print(f"Precision@10: {p10:.3f} (4 of top 10)")
print(f"Precision@20: {p20:.3f} (7 of top 20)")
print(f"Precision@50: {p50:.3f} (16 of top 50)")



Successfully generated and wrote ranked queue to: ../outputs/baseline_action_score.csv
Total rows exported: 30,000 | File size: 3,103,165 bytes

=== Baseline Rule Performance ===
Base Decline Rate: 54.2%
Precision@10: 0.400 (4 of top 10)
Precision@20: 0.350 (7 of top 20)
Precision@50: 0.320 (16 of top 50)


## 3. Top-20 review

### Detailed Line-by-Line Review of the Top 10 Recommendations
For each of our top ten highest-scored items, we document the assigned action, why it landed there, and what operational condition would make that recommendation wrong:

1. **Rank 1 (`content_69fad7e6c50c`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** High demand (28,000 impressions), Page 1 ranking (`avg_pos` 4.7), 106 days since update, and actively declining (`trend_direction == 'down'`).
   - **What would make it wrong:** If the drop was caused by temporary SERP snippet testing or broad seasonal query contraction rather than content obsolescence.

2. **Rank 2 (`content_9532f197bbc8`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** Massive head-term demand (309,192 impressions), top ranking (`avg_pos` 2.0), 104 days since update, actively trending down.
   - **What would make it wrong:** If Google rolled out an AI Overview or knowledge card capturing user clicks without an actual drop in organic ranking position.

3. **Rank 3 (`content_03d2673b2553`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** Massive visibility (143,314 impressions) and strong position (`avg_pos` 1.9) with 104 days since update.
   - **What would make it wrong:** **Observed trend is STABLE.** Editing an already-stable position-1 URL risks disrupting keyword rankings and introducing needless volatility.

4. **Rank 4 (`content_7a6df559322d`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 43,650 impressions, position 0.7, actively declining (`down`).
   - **What would make it wrong:** Average position < 1.0 indicates a sitelink or measurement artifact where standard content refresh cannot alter rank.

5. **Rank 5 (`content_6ac3ab740bbf`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 22,462 impressions, mid-Page 1 position (`avg_pos` 4.6), actively declining (`down`).
   - **What would make it wrong:** If competing articles expanded commercial intent while this article is strictly informational.

6. **Rank 6 (`content_4a6607efcb46`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** Top ranking (`avg_pos` 2.2) and 128,068 impressions with 104 days since update.
   - **What would make it wrong:** **Observed trend is UP.** Modifying an article that is actively gaining momentum can disrupt positive search engine re-evaluation.

7. **Rank 7 (`content_654d006adc44`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 131,328 impressions at position 2.4.
   - **What would make it wrong:** **Observed trend is STABLE.** The page is an evergreen anchor that does not need rewriting.

8. **Rank 8 (`content_fd1dc2828b88`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 46,739 impressions at position 1.7.
   - **What would make it wrong:** **Observed trend is STABLE.** Modifying stable high-ranking URLs burns editorial hours without incremental return.

9. **Rank 9 (`content_4d1fe5b32dc2`):**
   - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 97,999 impressions at position 2.5 with 104 days since update.
   - **What would make it wrong:** **Observed trend is STABLE.** Over-weighting search volume over true decay momentum.

10. **Rank 10 (`content_140e1efff17e`):**
    - **Action:** `refresh` | **Reason Code:** `page_one_decay_risk`
    - **Why it's there:** 44,437 impressions at position 1.7.
    - **What would make it wrong:** **Observed trend is STABLE.** Stable performer falsely flagged by volume-heavy rules.


In [3]:
# Preview Top 20 Ranked Items from the Baseline Queue
top20_preview = df_ranked.head(20)[[
    "rank", "content_id", "client_id", "baseline_score", "reason_code", 
    "action_label", "impressions_90d", "avg_position", "days_since_last_update", 
    "trend_direction"
]]

print("=== Top 20 Ranked Baseline Queue Preview ===")
print(top20_preview.to_string(index=False))



=== Top 20 Ranked Baseline Queue Preview ===
 rank           content_id         client_id  baseline_score         reason_code action_label  impressions_90d  avg_position  days_since_last_update trend_direction
    1 content_69fad7e6c50c client_7f2253d7e2           95.54 page_one_decay_risk      refresh            28000           4.7                     106            down
    2 content_9532f197bbc8 client_4e07408562           95.35 page_one_decay_risk      refresh           309192           2.0                     104            down
    3 content_03d2673b2553 client_19581e27de           95.34 page_one_decay_risk      refresh           143314           1.9                     104          stable
    4 content_7a6df559322d client_19581e27de           95.25 page_one_decay_risk      refresh            43650           0.7                     104            down
    5 content_6ac3ab740bbf client_f369cb89fc           95.15 page_one_decay_risk      refresh            22462           4.6      

## 4. Weak picks + leakage check

### Analysis of Weak Picks in the Top Queue
A critical inspection of our top queue reveals the primary limitation of hand-written rules:
- **False Alarms on Stable / Growing Pages:**
  In the top 10, **6 out of 10 pages** are not actually declining (Rank 6 is trending `up`, and Ranks 3, 7, 8, 9, 10 are `stable`).
- **Why the Rule Failed:**
  The rule relies heavily on impression volume and position tiers. It implicitly assumes that any high-exposure page older than 90 days needs a refresh. In reality, many mature high-volume pages are stable, evergreen performers. Recommending refreshes for these URLs wastes scarce editorial hours and risks breaking keyword anchors.
- **The Machine Learning Opportunity (Week 5):**
  A learned model will integrate non-linear interaction terms (e.g. CTR decay relative to position expected CTR, scroll retention, impression velocity) to suppress stable head pages and prioritize genuine decay.

---

### Strict Anti-Leakage Audit
We verify that no future information or label-derived signals were used:
- Features used: `impressions_90d`, `avg_position`, `days_since_last_update`. All are observable strictly within the trailing observation window.
- Excluded: `trend_direction`, `trend_pct`, 30-day recent windows, forward-window metrics, and product decision flags.
- Zero circular leakage exists in this baseline.


In [4]:
# Weak picks audit: identify non-declining items in top 50
top50 = df_ranked.head(50)
weak_picks = top50[top50["trend_direction"].isin(["stable", "up"])]

print(f"=== Weak Picks Audit (Top 50 Queue) ===")
print(f"Total false alarms in top 50: {len(weak_picks)} / 50 ({len(weak_picks)/50:.1%})")
print(f"  - Stable pages falsely flagged: {(weak_picks['trend_direction'] == 'stable').sum()}")
print(f"  - Up-trending pages falsely flagged: {(weak_picks['trend_direction'] == 'up').sum()}")

# Save run receipts to work/outputs/baseline_metrics.json
metrics_receipt = {
    "assignment": "ML-07",
    "lane": "Refresh / Content Opportunity Scoring",
    "total_scored_rows": len(df_ranked),
    "base_rate": float(base_rate),
    "precision_at_10": float(p10),
    "precision_at_20": float(p20),
    "precision_at_50": float(p50),
    "top_50_false_alarms": int(len(weak_picks)),
    "output_csv": "work/outputs/baseline_action_score.csv",
    "leakage_audit": "PASSED (Zero label-derived or future features used)"
}

receipt_path = out_dir / "baseline_metrics.json"
with open(receipt_path, "w") as f:
    json.dump(metrics_receipt, f, indent=2)

print(f"\nSaved metrics receipt to: {receipt_path}")



=== Weak Picks Audit (Top 50 Queue) ===
Total false alarms in top 50: 34 / 50 (68.0%)
  - Stable pages falsely flagged: 33
  - Up-trending pages falsely flagged: 1

Saved metrics receipt to: ../outputs/baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
